In [18]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('User Defined Function').getOrCreate()

In [26]:
columns = ['Seqno' , 'Name']
data = [('1','Fares Ashraf'), ('2','tracey smith'),('3', 'amy sandres')]

df = spark.createDataFrame(data = data, schema=columns)

In [54]:
def convertCase(str):
    resStr=""
    arr = str.split(" ")
    #print(arr)
    for x in arr:
        resStr = resStr + x[0:1].upper() + x[1:len(x)] + " "
        #print(resStr)
        return resStr

In [38]:
convertCase('fares ashraf')

'Fares '

In [41]:
df.select(col('Seqno')), convertCase(df['Name'].alias('Name')).show()

TypeError: 'Column' object is not callable

In [20]:
#UDF <-- Work on Columns With distributed Way

In [22]:
from pyspark.sql.functions import udf , col
from pyspark.sql.types import StructType , StructField , IntegerType , FloatType , StringType

In [45]:
convertUDF = udf(convertCase , StringType())

In [49]:
df.select(col('Seqno'), \
    convertUDF(col('Name')).alias('Name')).show()

+-----+-------+
|Seqno|   Name|
+-----+-------+
|    1| Fares |
|    2|Tracey |
|    3|   Amy |
+-----+-------+



In [53]:
@udf(returnType=StringType())
def convertCase_(str):
    resStr=""
    arr = str.split(" ")
    #print(arr)
    for x in arr:
        resStr = resStr + x[0:1].upper() + x[1:len(x)] + " "
        #print(resStr)
        return resStr

In [55]:
""" Using UDF On SQL """
spark.udf.register("convertUDFSQL" , convertCase , StringType())

<function __main__.convertCase(str)>

In [56]:
df.createOrReplaceTempView('NAME_TABLE')

In [58]:
spark.sql("select Seqno  , convertUDFSQL(Name) as Name from NAME_TABLE").show()

+-----+-------+
|Seqno|   Name|
+-----+-------+
|    1| Fares |
|    2|Tracey |
|    3|   Amy |
+-----+-------+

